# Cross-encoder (D-028, large) — BAAI/bge-reranker-v2-m3 (Apache-2.0, 568M params), fine-tuned only on provided train data
Kaggle: add the dataset with train.parquet / score_*.parquet as input; Settings → Accelerator **GPU T4 x2**, **Internet on**.
Run with **Save Version → Save & Run All (Commit)** so it runs in the background; download `/kaggle/working/out/*.parquet`.

In [ ]:
import os, glob, time, math, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
src = glob.glob('/kaggle/input/*/train.parquet') + glob.glob('/kaggle/input/*/*/train.parquet')
assert src, 'add the bundle dataset as input'
BUNDLE = os.path.dirname(src[0]); OUT = '/kaggle/working/out'; os.makedirs(OUT, exist_ok=True)
print(BUNDLE, os.listdir(BUNDLE), torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
BASE, MAXLEN, SEED = 'BAAI/bge-reranker-v2-m3', 80, 42
MAX_TRAIN, BS, LR, TRAIN_MINUTES = 1_000_000, 96, 2e-5, 170   # time guard keeps ~1h for scoring
torch.manual_seed(SEED); np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1).cuda()
net = torch.nn.DataParallel(model) if torch.cuda.device_count() > 1 else model
tr = pd.read_parquet(f'{BUNDLE}/train.parquet').sample(frac=1.0, random_state=SEED).reset_index(drop=True)
dev, tr = tr.iloc[:10000].reset_index(drop=True), tr.iloc[10000:10000 + MAX_TRAIN].reset_index(drop=True)
print('train', len(tr), 'pos rate', round(tr.label.mean(), 4), 'dev', len(dev), flush=True)

In [ ]:
def batches(df, bs, shuffle, seed=0):
    idx = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for s in range(0, len(df), bs):
        j = idx[s:s + bs]
        enc = tok(df.ta.values[j].tolist(), df.tb.values[j].tolist(), truncation=True, max_length=MAXLEN,
                  padding=True, return_tensors='pt')
        yield j, {k: v.cuda(non_blocking=True) for k, v in enc.items()}

def evaluate(df, bs=1024):
    net.eval(); out = np.zeros(len(df), dtype=np.float32)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for j, enc in batches(df, bs, False):
            out[j] = net(**enc).logits.squeeze(-1).float().cpu().numpy()
    net.train(); return out

steps = math.ceil(len(tr) / BS)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 500) * max(0.0, 1 - s / steps))
scaler = torch.amp.GradScaler('cuda')
y_all = torch.tensor(tr.label.values, dtype=torch.float32)
t0 = time.time(); net.train()
for step, (j, enc) in enumerate(batches(tr, BS, True, SEED), 1):
    with torch.autocast('cuda', dtype=torch.float16):
        logit = net(**enc).logits.squeeze(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logit.float(), y_all[j].cuda())
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
    el = time.time() - t0
    if step % 500 == 0:
        print(f'step {step}/{steps} loss {loss.item():.4f} {el/60:.1f} min, eta {el/step*(steps-step)/60:.1f} min', flush=True)
    if el > TRAIN_MINUTES * 60:
        print(f'time guard: stopping at step {step}/{steps}', flush=True); break
d = evaluate(dev); p = 1 / (1 + np.exp(-d)); yv = dev.label.values
print('dev logloss', round(float(-np.mean(yv*np.log(np.clip(p,1e-7,1)) + (1-yv)*np.log(np.clip(1-p,1e-7,1)))), 4),
      'acc', round(float(np.mean((p > 0.5) == yv)), 4), flush=True)

In [ ]:
for g in ('tune', 'holdout', 'test'):
    df = pd.read_parquet(f'{BUNDLE}/score_{g}.parquet')
    order = np.argsort((df.ta.str.len() + df.tb.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    t0 = time.time(); ce = evaluate(df)
    pd.DataFrame({'lid': df.lid, 'rid': df.rid, 'ce': ce}).to_parquet(f'{OUT}/ce_{g}.parquet', index=False)
    print(g, len(df), f'{(time.time()-t0)/60:.1f} min', flush=True)
print('DONE — download /kaggle/working/out/ce_tune.parquet, ce_holdout.parquet, ce_test.parquet')